# 16 — Management Consulting Cost Optimizer
> You describe a company's operations in plain text. The agent reads it and returns a full cost optimization report — quick wins first, each with a rationale, concrete steps, and an estimated annual saving.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q langchain-openai langchain-core pydantic python-dotenv
import os
os.environ['OPENAI_API_KEY'] = 'sk-...'  # replace

In [ ]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI


class Recommendation(BaseModel):
    title: str = Field(description="Short descriptive name for this recommendation")
    category: Literal["process", "technology", "people", "structure"]
    effort: Literal["low", "medium", "high"]
    impact: Literal["low", "medium", "high"]
    quadrant: Literal["quick_win", "major_project", "fill_in", "thankless_task"] = Field(
        description=(
            "quick_win=low effort+high impact, major_project=high effort+high impact, "
            "fill_in=low effort+low impact, thankless_task=high effort+low impact"
        )
    )
    estimated_annual_saving: Optional[str] = Field(
        default=None,
        description="Estimated annual saving in GBP where quantifiable, e.g. 'GBP 180k'",
    )
    rationale: str = Field(description="Why this is an inefficiency and what fixing it achieves")
    implementation_steps: List[str] = Field(
        description="Concrete ordered steps to implement this recommendation"
    )


class CostOptimizationReport(BaseModel):
    """Management consulting cost optimization assessment with 2x2 effort-impact ranking."""

    company: Optional[str] = None
    total_addressable_saving: Optional[str] = Field(
        default=None,
        description="Sum of all quantified annual savings across all recommendations",
    )
    executive_summary: str = Field(
        description="3-4 sentences for a C-suite audience: what was found and the priority action"
    )
    quick_wins: List[Recommendation] = Field(
        description="Low effort, high impact -- highest ROI, do first"
    )
    major_projects: List[Recommendation] = Field(
        description="High effort, high impact -- worth doing, need a project plan"
    )
    fill_ins: List[Recommendation] = Field(
        description="Low effort, low impact -- do when bandwidth allows"
    )
    thankless_tasks: List[Recommendation] = Field(
        description="High effort, low impact -- avoid or deprioritise"
    )
    prioritization_note: str = Field(
        description="Guidance on sequencing: which quick win to tackle first and why"
    )


CONSULTANT_SYSTEM = SystemMessage(
    """You are a management consultant conducting a cost optimization review.

Analyse the operational profile provided and identify inefficiencies. For each one:

1. Classify by EFFORT to fix: low / medium / high
2. Classify by SAVINGS POTENTIAL (impact): low / medium / high
3. Assign the correct 2x2 quadrant using this exact logic:
   - effort=low  + impact=high  --> quick_win
   - effort=high + impact=high  --> major_project
   - effort=low  + impact=low   --> fill_in
   - effort=high + impact=low   --> thankless_task
   Note: treat \"medium\" effort as \"low\" for quadrant purposes; treat \"medium\" impact
   as \"high\" for quadrant purposes. The goal is to bias toward action.
4. Estimate annual saving in GBP where the data supports it.
5. List concrete implementation steps (3-5 steps).

Sort quick_wins first -- they are the client's highest ROI actions.
Provide an executive_summary of 3-4 sentences and a prioritization_note."""
)


def run(operational_profile: str) -> CostOptimizationReport:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    consultant = CONSULTANT_SYSTEM | llm.with_structured_output(CostOptimizationReport)
    return consultant.invoke(
        HumanMessage(content="Operational profile:\n\n" + operational_profile)
    )


print("Ready.")

## The scenario

We are running a cost review for a 150-person professional services firm that grew quickly through acquisition and never consolidated its back-office operations. Finance, HR, and IT each have their own toolsets, workflows, and supplier contracts — much of it duplicated or manual. The agent will diagnose every inefficiency, assign each one to a quadrant, and tell the leadership team exactly where to start.

In [ ]:
SAMPLE_PROFILE = """
COMPANY: Ashford & Vale Partners LLP
Employees: 150  |  Revenue: GBP 18m  |  Sector: Professional services (accountancy and advisory)

IDENTIFIED OPERATIONAL ISSUES:

1. Duplicate payroll and HR systems
   - Two separate HR platforms in use (legacy from 2021 acquisition); both maintained in parallel
   - HR team of 6 manually reconciles employee records monthly across both systems
   - Combined annual licensing: GBP 95k; estimated 50% redundancy
   - Reconciliation takes approximately 3 days per month

2. No centralised procurement for professional subscriptions
   - Partners independently renew research databases, legal reference tools, and CPD platforms
   - 23 active subscriptions identified; 9 have direct feature overlap
   - Total annual spend: GBP 210k; estimated 35% could be consolidated or eliminated
   - No approval workflow; invoices arrive ad hoc to finance

3. Manual timesheet and billing process
   - Consultants log time in spreadsheets; finance team re-enters into billing system weekly
   - 2 FTE in finance dedicated to timesheet chase and data entry
   - Average billing cycle: 22 days from month end (peer benchmark: 8 days)
   - Late billing estimated to delay GBP 1.2m in cash flow per month

4. Office space underutilisation post-hybrid shift
   - Three office locations at combined lease cost of GBP 420k per annum
   - Average occupancy since 2022: 38% across all sites
   - One location (Reading) averages under 20% occupancy; lease expires in 14 months
   - No formal desk-booking system; space planning done by facilities manager manually

5. Paper-based client onboarding and KYC
   - New client onboarding requires printed forms, wet signatures, and postal ID verification
   - Average onboarding time: 18 days; losing an estimated 2 clients per quarter due to friction
   - KYC compliance team spends 40% of time on manual document chasing
   - At average client value of GBP 35k, lost clients = GBP 280k annual revenue impact

6. Fragmented IT support contracts
   - 4 separate managed service agreements across 3 vendors (acquisition legacy)
   - Annual IT support spend: GBP 185k; benchmarked at GBP 110k for a firm this size
   - Contracts renew at different dates; no consolidated SLA or account management
   - Two contracts overlap on endpoint security coverage
"""

report = run(SAMPLE_PROFILE)

company = report.company or "Ashford & Vale Partners LLP"
print("=" * 65)
print(f"COST OPTIMIZATION REPORT | {company}")
print("=" * 65)
print(f"\n{report.executive_summary}")

if report.total_addressable_saving:
    print(f"\nTotal addressable saving: {report.total_addressable_saving}")


def print_recs(label: str, recs: list) -> None:
    if not recs:
        return
    print(f"\n{label} ({len(recs)}):")
    for r in recs:
        saving = f" | Saving: {r.estimated_annual_saving}" if r.estimated_annual_saving else ""
        print(f"\n  [{r.effort.upper()} effort / {r.impact.upper()} impact]{saving}")
        print(f"  {r.title} [{r.category}]")
        print(f"  {r.rationale}")
        print("  Steps:")
        for step in r.implementation_steps:
            print(f"    - {step}")


print_recs("QUICK WINS", report.quick_wins)
print_recs("MAJOR PROJECTS", report.major_projects)
print_recs("FILL-INS", report.fill_ins)
print_recs("THANKLESS TASKS (avoid)", report.thankless_tasks)

print(f"\nPRIORITIZATION: {report.prioritization_note}")

## Use your own data

Replace the input above with:
- A plain-text description of your company's operations (headcount, revenue, sector, known pain points)
- Specific cost figures where you have them — the more concrete the numbers, the sharper the saving estimates

The agent will return every inefficiency ranked by effort and impact, with a prioritized action plan and estimated annual saving for each.